# Protein Annotation Toolkit - Tutorial

This notebook demonstrates the key features of the Protein Annotation Toolkit, a bioinformatics tool for protein data management.

## Features Demonstrated

1. **Biological Identifier Validation** - Validate UniProt IDs with detailed error messages
2. **XML Parsing** - Parse UniProt and BLAST XML formats
3. **API Integration** - Fetch data from UniProt and KEGG REST APIs  
4. **Data Visualization** - Display protein information in formatted tables

## Requirements

```bash
pip install protein-annotation-toolkit jupyter
```

## 1. Import Required Modules

In [ ]:
from pathlib import Path
import pandas as pd
from rich.console import Console
from rich.table import Table

# Import toolkit components
from protein_annotation_toolkit.validators import validate_uniprot_id, validate_uniprot_ids
from protein_annotation_toolkit.parsers import UniProtXMLParser, BlastXMLParser
from protein_annotation_toolkit.clients import UniProtClient, KEGGClient

# Initialize Rich console for pretty output
console = Console()

print("✓ Imports successful")

## 2. Biological Identifier Validation

UniProt IDs follow a specific format: **one letter followed by exactly 5 digits** (e.g., P13773).

The toolkit provides detailed validation with specific error messages.

In [ ]:
# Test various UniProt ID formats
test_ids = [
    "P13773",      # Valid
    "Q02293",      # Valid
    "INVALID",     # Invalid - not enough digits
    "P1234",       # Invalid - only 4 digits
    "12345A",      # Invalid - starts with number
    "P1234!",      # Invalid - special character
]

print("\n" + "="*60)
print("UniProt ID Validation")
print("="*60 + "\n")

for uid in test_ids:
    is_valid, error_msg = validate_uniprot_id(uid)
    status = "✓ VALID" if is_valid else f"✗ INVALID: {error_msg}"
    print(f"{uid:15} → {status}")

## 3. Parsing UniProt XML Files

The toolkit can parse UniProt XML format to extract:
- Protein metadata (accession, name, organism)
- Amino acid sequence
- Gene Ontology (GO) terms
- PDB structure cross-references

In [ ]:
# Parse a UniProt XML file
parser = UniProtXMLParser()
xml_file = Path("data/uniprot_xml/P13773.xml")

if xml_file.exists():
    data = parser.parse_file(xml_file)
    
    print("\n" + "="*60)
    print(f"Protein: {data['accession']}")
    print("="*60)
    print(f"Entry Name:       {data['entry_name']}")
    print(f"Protein Name:     {data['recommended_name']}")
    print(f"Organism:         {data['organism']}")
    print(f"Taxonomy ID:      {data['taxonomy_id']}")
    print(f"Sequence Length:  {data['sequence_length']} amino acids")
    print(f"GO Terms:         {len(data['go_terms'])}")
    print(f"PDB Structures:   {len(data['pdb_crossrefs'])}")
else:
    print(f"⚠ XML file not found: {xml_file}")
    print("Run the data fetching cell first!")

### 3.1 View GO Term Annotations

Gene Ontology terms provide functional annotations.

In [ ]:
if xml_file.exists() and data['go_terms']:
    # Convert to DataFrame for nice display
    go_df = pd.DataFrame(data['go_terms'])
    go_df.columns = ['GO ID', 'Term']
    
    print(f"\nGO Terms for {data['accession']} (showing first 15):\n")
    display(go_df.head(15))
    
    # Count by ontology prefix
    print("\nGO Term Categories:")
    for category in ['P:', 'F:', 'C:']:
        count = sum(1 for term in data['go_terms'] if term['term'].startswith(category))
        category_name = {
            'P:': 'Biological Process',
            'F:': 'Molecular Function',
            'C:': 'Cellular Component'
        }[category]
        print(f"  {category_name:20} {count:3} terms")

### 3.2 View Sequence Information

In [ ]:
if xml_file.exists() and data['sequence']:
    sequence = data['sequence']
    
    print(f"\nAmino Acid Sequence ({len(sequence)} residues):\n")
    
    # Display sequence in blocks of 60
    for i in range(0, min(len(sequence), 240), 60):
        print(f"{i+1:4d}  {sequence[i:i+60]}")
    
    if len(sequence) > 240:
        print(f"     ... ({len(sequence) - 240} more residues)")
    
    # Calculate basic composition
    from collections import Counter
    composition = Counter(sequence)
    
    print("\nAmino Acid Composition (top 5):")
    for aa, count in composition.most_common(5):
        percentage = (count / len(sequence)) * 100
        print(f"  {aa}: {count:3d} ({percentage:5.2f}%)")

## 4. Fetching Data from UniProt API

The toolkit provides async clients for fetching data from external APIs.

**Note:** This requires an active internet connection.

In [ ]:
import asyncio

async def fetch_protein_data(uniprot_id: str):
    """Fetch protein data from UniProt API."""
    async with UniProtClient() as client:
        print(f"Fetching {uniprot_id} from UniProt...")
        xml_content = await client.fetch_xml(uniprot_id)
        
        # Parse the fetched XML
        parser = UniProtXMLParser()
        data = parser.parse_string(xml_content)
        
        return data

# Fetch a protein
protein_id = "P29274"  # Adenylate cyclase A
fetched_data = await fetch_protein_data(protein_id)

print(f"\n✓ Successfully fetched {fetched_data['accession']}")
print(f"  Protein: {fetched_data['recommended_name']}")
print(f"  Organism: {fetched_data['organism']}")
print(f"  Length: {fetched_data['sequence_length']} aa")

### 4.1 Batch Fetching

The toolkit supports concurrent batch fetching for efficiency.

In [ ]:
async def fetch_multiple_proteins(uniprot_ids: list):
    """Fetch multiple proteins concurrently."""
    async with UniProtClient() as client:
        print(f"Fetching {len(uniprot_ids)} proteins concurrently...")
        results = await client.fetch_xml_batch(uniprot_ids)
        
        # Parse all results
        parser = UniProtXMLParser()
        parsed_data = {}
        
        for uid, xml_content in results.items():
            if xml_content:
                try:
                    data = parser.parse_string(xml_content)
                    parsed_data[uid] = data
                except Exception as e:
                    print(f"  ✗ Failed to parse {uid}: {e}")
            else:
                print(f"  ✗ Failed to fetch {uid}")
        
        return parsed_data

# Fetch multiple proteins
protein_ids = ["P13773", "P29274", "P41595"]
batch_results = await fetch_multiple_proteins(protein_ids)

print(f"\n✓ Successfully fetched {len(batch_results)} proteins")

# Display summary
summary_data = []
for uid, data in batch_results.items():
    summary_data.append({
        'UniProt ID': uid,
        'Protein Name': data['recommended_name'][:50],
        'Organism': data['organism'][:30],
        'Length': data['sequence_length'],
        'GO Terms': len(data['go_terms'])
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

## 5. KEGG Pathway Integration

The toolkit can fetch KEGG pathway annotations for proteins.

In [ ]:
async def get_protein_pathways(uniprot_id: str):
    """Get KEGG pathways for a protein."""
    async with KEGGClient() as client:
        print(f"Fetching KEGG pathways for {uniprot_id}...")
        pathways = await client.get_pathways_for_protein(uniprot_id)
        return pathways

# Get pathways for a protein
pathways = await get_protein_pathways("P13773")

if pathways:
    print(f"\n✓ Found {len(pathways)} KEGG pathways:\n")
    
    pathway_df = pd.DataFrame(pathways)
    pathway_df.columns = ['Pathway ID', 'Name', 'Class']
    display(pathway_df)
else:
    print("\nNo KEGG pathways found or protein not mapped to KEGG.")

## 6. BLAST XML Parsing

The toolkit can parse BLAST XML output to extract search results.

In [ ]:
# Parse BLAST XML
blast_parser = BlastXMLParser()
blast_xml = Path("data/blast/web_blast_output.xml")

if blast_xml.exists():
    blast_data = blast_parser.parse_file(blast_xml, top_n=10)
    
    print("\n" + "="*60)
    print("BLAST Search Results")
    print("="*60)
    print(f"Program:      {blast_data['program']}")
    print(f"Database:     {blast_data['database']}")
    print(f"Query:        {blast_data['query_def']}")
    print(f"Query Length: {blast_data['query_length']} aa")
    print(f"Total Hits:   {len(blast_data['hits'])}")
    
    # Get best hits
    best_hits = blast_parser.get_best_hits(
        blast_data,
        max_hits=5,
        max_e_value=1e-10
    )
    
    print(f"\nTop 5 Hits (E-value < 1e-10):\n")
    
    hit_data = []
    for hit in best_hits:
        best_hsp = hit['hsps'][0]  # Best HSP
        hit_data.append({
            'Accession': hit['hit_accession'],
            'Description': hit['hit_def'][:60],
            'E-value': f"{best_hsp['e_value']:.2e}",
            'Identity': f"{best_hsp['identity']}/{best_hsp['alignment_length']}",
            'Length': hit['hit_length']
        })
    
    hit_df = pd.DataFrame(hit_data)
    display(hit_df)
else:
    print(f"⚠ BLAST XML file not found: {blast_xml}")

## 7. Complete Workflow Example

Let's put it all together: fetch, parse, and analyze a protein.

In [ ]:
async def analyze_protein(uniprot_id: str):
    """Complete protein analysis workflow."""
    
    # Step 1: Validate ID
    is_valid, error = validate_uniprot_id(uniprot_id)
    if not is_valid:
        print(f"✗ Invalid UniProt ID: {error}")
        return
    
    print(f"\n{'='*70}")
    print(f"Protein Analysis: {uniprot_id}")
    print(f"{'='*70}\n")
    
    # Step 2: Fetch from UniProt
    print("[1/3] Fetching from UniProt...")
    async with UniProtClient() as uniprot_client:
        xml_content = await uniprot_client.fetch_xml(uniprot_id)
    
    # Step 3: Parse XML
    print("[2/3] Parsing protein data...")
    parser = UniProtXMLParser()
    data = parser.parse_string(xml_content)
    
    # Step 4: Get KEGG pathways
    print("[3/3] Fetching KEGG pathways...")
    async with KEGGClient() as kegg_client:
        pathways = await kegg_client.get_pathways_for_protein(uniprot_id)
    
    # Display results
    print(f"\n{'─'*70}")
    print("ANALYSIS RESULTS")
    print(f"{'─'*70}\n")
    
    print(f"Protein:          {data['recommended_name']}")
    print(f"Entry Name:       {data['entry_name']}")
    print(f"Organism:         {data['organism']}")
    print(f"Sequence Length:  {data['sequence_length']} aa")
    print(f"GO Annotations:   {len(data['go_terms'])} terms")
    print(f"PDB Structures:   {len(data['pdb_crossrefs'])} entries")
    print(f"KEGG Pathways:    {len(pathways)} pathways")
    
    return data, pathways

# Analyze a protein
protein_data, protein_pathways = await analyze_protein("Q02293")

## Summary

This tutorial demonstrated:

✅ **Biological identifier validation** with detailed error messages  
✅ **XML parsing** for UniProt and BLAST formats  
✅ **REST API integration** with UniProt and KEGG  
✅ **Concurrent batch operations** for performance  
✅ **Complete analysis workflows** combining multiple data sources

### Next Steps

- **Database Integration**: Store parsed data in PostgreSQL for querying
- **BLAST Submission**: Submit new BLAST searches programmatically
- **Data Export**: Export results to CSV, JSON, or other formats
- **Advanced Queries**: Complex database queries across multiple tables

See the [README](../README.md) for CLI usage and more examples.